In [1]:
import os
from pathlib import Path

os.chdir(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())

In [3]:
import pandas as pd

consensus = pd.read_csv('./results/DEG/consensus_DEGs.csv')
print(consensus.shape)
print(consensus.head())

(1285, 2)
       Gene direction
0  PAFAH1B3        Up
1     CENPV        Up
2      PLK4        Up
3    DUSP26        Up
4    PAXIP1        Up


In [5]:
hub_genes = {
    'Cluster1_CellCycle': [
        'KIF11', 'KIF2C', 'BUB1', 'ASPM', 'BUB1B', 'CDK1', 'CCNA2',
        'TPX2', 'DLGAP5', 'CENPF', 'EXO1', 'BRCA1', 'CHEK1',
        'TYMS', 'PLK1', 'CDC20', 'AURKB', 'TOP2A'
    ],
    'Cluster2_Immune': [
        'IL6', 'TLR4', 'TLR2', 'CCL2', 'CASP1', 'IL18',
        'TNFRSF1A', 'MYD88', 'IL1R1', 'TLR3', 'CD36', 'PTGS2'
    ],
    'Cluster3_Epigenetic': [
        'FOS', 'EGR1', 'EGR2', 'ATF3', 'MYC', 'FOSB', 'JUNB',
        'DUSP1', 'ZFP36', 'IER2', 'CEBPB', 'HDAC2', 'EZH2', 'CEBPD', 'CCND1'
    ],
    'Cluster4_ECM': [
        'CD44', 'ITGA3', 'ITGA8', 'ITGB5', 'LAMA2'
    ]
}

results = []
for cluster, genes in hub_genes.items():
    matched = consensus[consensus['Gene'].isin(genes)]
    for _, row in matched.iterrows():
        results.append({
            'Cluster': cluster,
            'Gene': row['Gene'],
            'Direction': row['direction']
        })
    not_found = set(genes) - set(matched['Gene'].tolist())
    if not_found:
        print(f"{cluster} — not found in consensus: {not_found}")

direction_df = pd.DataFrame(results)
print(direction_df.to_string())

                Cluster      Gene Direction
0    Cluster1_CellCycle     KIF11        Up
1    Cluster1_CellCycle     CHEK1        Up
2    Cluster1_CellCycle    DLGAP5        Up
3    Cluster1_CellCycle     TOP2A        Up
4    Cluster1_CellCycle     CENPF        Up
5    Cluster1_CellCycle      ASPM        Up
6    Cluster1_CellCycle      PLK1        Up
7    Cluster1_CellCycle     CCNA2        Up
8    Cluster1_CellCycle      EXO1        Up
9    Cluster1_CellCycle      TPX2        Up
10   Cluster1_CellCycle     BRCA1        Up
11   Cluster1_CellCycle      TYMS        Up
12   Cluster1_CellCycle     CDC20        Up
13   Cluster1_CellCycle      CDK1        Up
14   Cluster1_CellCycle     BUB1B        Up
15   Cluster1_CellCycle      BUB1        Up
16   Cluster1_CellCycle     AURKB        Up
17   Cluster1_CellCycle     KIF2C        Up
18      Cluster2_Immune       IL6      Down
19      Cluster2_Immune  TNFRSF1A      Down
20      Cluster2_Immune      CCL2      Down
21      Cluster2_Immune      TLR

In [7]:
print("=" * 50)
print("DIRECTIONAL PURITY SUMMARY PER CLUSTER")
print("=" * 50)

for cluster in hub_genes.keys():
    subset = direction_df[direction_df['Cluster'] == cluster]
    total = len(subset)
    up = len(subset[subset['Direction'] == 'Up'])
    down = len(subset[subset['Direction'] == 'Down'])
    dominant = 'Up' if up > down else 'Down'
    purity = max(up, down) / total * 100
    
    print(f"\n{cluster}:")
    print(f"  Total hub genes in consensus: {total}/{len(hub_genes[cluster])}")
    print(f"  Up: {up} | Down: {down}")
    print(f"  Dominant direction: {dominant}")
    print(f"  Directional purity: {purity:.1f}%")

DIRECTIONAL PURITY SUMMARY PER CLUSTER

Cluster1_CellCycle:
  Total hub genes in consensus: 18/18
  Up: 18 | Down: 0
  Dominant direction: Up
  Directional purity: 100.0%

Cluster2_Immune:
  Total hub genes in consensus: 12/12
  Up: 0 | Down: 12
  Dominant direction: Down
  Directional purity: 100.0%

Cluster3_Epigenetic:
  Total hub genes in consensus: 15/15
  Up: 2 | Down: 13
  Dominant direction: Down
  Directional purity: 86.7%

Cluster4_ECM:
  Total hub genes in consensus: 5/5
  Up: 0 | Down: 5
  Dominant direction: Down
  Directional purity: 100.0%


In [9]:
direction_df.to_csv('./results/DEG/hub_gene_directional_verification.csv', index=False)
print("Saved hub gene directional verification results")
print(f"\nTotal hub genes verified: {len(direction_df)}")
print(f"All hub genes found in consensus: {len(direction_df) == sum(len(g) for g in hub_genes.values())}")

Saved hub gene directional verification results

Total hub genes verified: 50
All hub genes found in consensus: True


In [11]:
import os
os.makedirs('./results/DEG/hub_gene_lists', exist_ok=True)

for cluster, genes in hub_genes.items():
    pd.Series(genes).to_csv(
        f'./results/DEG/hub_gene_lists/{cluster}_hub_genes.csv',
        index=False,
        header=['Gene']
    )
print("Saved per-cluster hub gene list files")

Saved per-cluster hub gene list files
